# 03 - Text Preprocessing and Embeddings

Normalises dates and legal text, removes stop words, lemmatises content, segments sentences, and creates sentence-transformer embeddings.

**Expected input:** `data/processed/wto_cases.csv`  
**Generated output:** `data/processed/preprocessed_wto_cases.csv` and NumPy embedding files

> Install dependencies once from the repository root with `pip install -r requirements.txt`. Run the notebooks in numerical order.

In [ ]:
from pathlib import Path

# Resolve repository paths whether Jupyter starts in the repository root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_PDF_DIR = DATA_DIR / "raw" / "wto_case_pdfs"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = PROCESSED_DIR / "wto_cases.csv"
PROCESSED_DATA_PATH = PROCESSED_DIR / "preprocessed_wto_cases.csv"

print(f"Project root: {PROJECT_ROOT}")


Install the spaCy English model once before running this notebook:

```bash
python -m spacy download en_core_web_sm
```

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [3]:
import pandas as pd

# Load the dataset
file_path = RAW_DATA_PATH  
df = pd.read_csv(file_path)

# Display the first few rows
df.head()


,Case ID,Title,Complainant,Respondent,Date,Articles,Summary
0,ds491,US – COATED PAPER (INDONESIA)1,Indonesia,United States,22 January 2018,"ADA Arts. 3.5, 3.7, 3.8;\nASCM Arts. 2.1(c), 1...",FINDINGS2\n• ASCM Art. 14(d) (rejection of in-...
1,ds33,US – WOOL SHIRTS AND BLOUSES1,India,United States,23 May 1997,ATC Arts. 6 and 2.4,/AB FINDINGS\n• ATC Art. 6 (transitional safeg...
2,ds403,PHILIPPINES – DISTILLED SPIRITS1,"European Union,\nUnited States",Philippines,20 January 2012,"GATT Art. III:2, first and\nsecond sentences",/AB FINDINGS\n• GATT Art. III:2 (national trea...
3,ds267A,US – UPLAND COTTON (ARTICLE 21.5 – BRAZIL)1,Brazil,United States,20 June 2008,"ASCM Arts.3, 5(c), 6.3(c), and item\n(j) of th...","/AB FINDINGS\n• AA Arts. 10.1 and 8, and ASCM ..."
4,ds44,JAPAN – FILM1,United States,Japan,22 April 1998,"GATT Arts. XXIII:1(b), III:4 and X:1",FINDINGS\n• GATT Art. XXIII:1(b) (non-violatio...


In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv(RAW_DATA_PATH)

# Convert date to standard format
df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.strftime("%Y-%m-%d")

# Replace blank date cells with 'Unknown'
df['Date'] = df['Date'].fillna('Unknown')

# Save the modified dataset to a new CSV file
df.to_csv(PROCESSED_DATA_PATH, index=False)

# Display the first few rows
df.head()


,Case ID,Title,Complainant,Respondent,Date,Articles,Summary
0,ds491,US – COATED PAPER (INDONESIA)1,Indonesia,United States,2018-01-22,"ADA Arts. 3.5, 3.7, 3.8;\nASCM Arts. 2.1(c), 1...",FINDINGS2\n• ASCM Art. 14(d) (rejection of in-...
1,ds33,US – WOOL SHIRTS AND BLOUSES1,India,United States,1997-05-23,ATC Arts. 6 and 2.4,/AB FINDINGS\n• ATC Art. 6 (transitional safeg...
2,ds403,PHILIPPINES – DISTILLED SPIRITS1,"European Union,\nUnited States",Philippines,2012-01-20,"GATT Art. III:2, first and\nsecond sentences",/AB FINDINGS\n• GATT Art. III:2 (national trea...
3,ds267A,US – UPLAND COTTON (ARTICLE 21.5 – BRAZIL)1,Brazil,United States,2008-06-20,"ASCM Arts.3, 5(c), 6.3(c), and item\n(j) of th...","/AB FINDINGS\n• AA Arts. 10.1 and 8, and ASCM ..."
4,ds44,JAPAN – FILM1,United States,Japan,1998-04-22,"GATT Arts. XXIII:1(b), III:4 and X:1",FINDINGS\n• GATT Art. XXIII:1(b) (non-violatio...


In [3]:
import re
import contractions
import pandas as pd

# Load the dataset
df = pd.read_csv(RAW_DATA_PATH)

# Function to normalize legal references
def normalize_legal_terms(text):
    text = re.sub(r"\bArt\s*(\d+)\s*sec\.?\s*(\d+)\b", r"Article \1, Section \2", text)
    return text

# Function to normalize text
def normalize_text(text):
    if pd.isnull(text) or not isinstance(text, str):
        return ""
    text = contractions.fix(text)  # Expand contractions
    text = text.lower()  # Convert to lowercase
    text = normalize_legal_terms(text)  # Standardize legal references
    text = re.sub(r"[^\w\s]", "", text)  # Remove punctuation & special characters
    return text

# Apply to Title, Summary, and Articles
df["Title_normalized"] = df["Title"].apply(normalize_text)
df["Summary_normalized"] = df["Summary"].apply(normalize_text)
df["Articles_normalized"] = df["Articles"].apply(normalize_text)

# Print results
print(df[["Title", "Title_normalized", "Articles", "Articles_normalized", "Summary", "Summary_normalized"]].head())


                                         Title  \
0               US – COATED PAPER (INDONESIA)1   
1                US – WOOL SHIRTS AND BLOUSES1   
2             PHILIPPINES – DISTILLED SPIRITS1   
3  US – UPLAND COTTON (ARTICLE 21.5 – BRAZIL)1   
4                                JAPAN – FILM1   

                         Title_normalized  \
0             us  coated paper indonesia1   
1            us  wool shirts and blouses1   
2         philippines  distilled spirits1   
3  us  upland cotton article 215  brazil1   
4                            japan  film1   

                                            Articles  \
0  ADA Arts. 3.5, 3.7, 3.8;\nASCM Arts. 2.1(c), 1...   
1                                ATC Arts. 6 and 2.4   
2       GATT Art. III:2, first and\nsecond sentences   
3  ASCM Arts.3, 5(c), 6.3(c), and item\n(j) of th...   
4               GATT Arts. XXIII:1(b), III:4 and X:1   

                                 Articles_normalized  \
0  ada arts 35 37 38\nascm arts 21c

In [4]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download('punkt_tab')
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

# Function for tokenization & stopword removal
def tokenize_and_remove_stopwords(text):
    tokens = word_tokenize(text)  # Tokenize text
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    return " ".join(tokens)

# Apply to normalized text
df["Title_tokenized"] = df["Title_normalized"].apply(tokenize_and_remove_stopwords)
df["Summary_tokenized"] = df["Summary_normalized"].apply(tokenize_and_remove_stopwords)
df["Articles_tokenized"] = df["Articles_normalized"].apply(tokenize_and_remove_stopwords)

# Print results
print(df[["Title_tokenized", "Articles_tokenized", "Summary_tokenized"]].head())


                        Title_tokenized  \
0            us coated paper indonesia1   
1               us wool shirts blouses1   
2        philippines distilled spirits1   
3  us upland cotton article 215 brazil1   
4                           japan film1   

                                  Articles_tokenized  \
0  ada arts 35 37 38 ascm arts 21c 127 14d 155 15...   
1                                      atc arts 6 24   
2               gatt art iii2 first second sentences   
3  ascm arts3 5c 63c item j illustrative list aa ...   
4                          gatt arts xxiii1b iii4 x1   

                                   Summary_tokenized  
0  findings2 ascm art 14d rejection incountry pri...  
1  ab findings atc art 6 transitional safeguard m...  
2  ab findings gatt art iii2 national treatment t...  
3  ab findings aa arts 101 8 ascm arts 31a 32 ill...  
4  findings gatt art xxiii1b nonviolation claim p...  


[nltk_data] Downloading package punkt to
[nltk_data]     <LOCAL_NLTK_DATA>...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     <LOCAL_NLTK_DATA>...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     <LOCAL_NLTK_DATA>...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
import spacy

nlp = spacy.load("en_core_web_sm")

# Function to lemmatize text
def lemmatize_text(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])

# Apply to tokenized text
df["Summary_lemmatized"] = df["Summary_tokenized"].apply(lemmatize_text)
df["Articles_lemmatized"] = df["Articles_tokenized"].apply(lemmatize_text)

# Print results
print(df[["Articles_lemmatized", "Summary_lemmatized"]].head())


                                 Articles_lemmatized  \
0  ada arts 35 37 38 ascm art 21c 127 14d 155 157...   
1                                      atc arts 6 24   
2                gatt art iii2 first second sentence   
3  ascm arts3 5c 63c item j illustrative list aa ...   
4                          gatt arts xxiii1b iii4 x1   

                                  Summary_lemmatized  
0  findings2 ascm art 14d rejection incountry pri...  
1  ab finding atc art 6 transitional safeguard me...  
2  ab findings gatt art iii2 national treatment t...  
3  ab findings aa arts 101 8 ascm art 31a 32 illu...  
4  findings gatt art xxiii1b nonviolation claim p...  


In [6]:
from nltk.tokenize import sent_tokenize

# Function for sentence segmentation
def segment_sentences(text):
    return sent_tokenize(text)

# Apply to lemmatized summary
df["Summary_segmented"] = df["Summary_lemmatized"].apply(segment_sentences)

# Print results
print(df[["Summary_lemmatized", "Summary_segmented"]].head(15))


                                   Summary_lemmatized  \
0   findings2 ascm art 14d rejection incountry pri...   
1   ab finding atc art 6 transitional safeguard me...   
2   ab findings gatt art iii2 national treatment t...   
3   ab findings aa arts 101 8 ascm art 31a 32 illu...   
4   findings gatt art xxiii1b nonviolation claim p...   
5   ab findings2 sps art 3 harmonization panel fin...   
6   ab findings aa art 91 c export subsidy payment...   
7   ab findings ada arts 31 32 34 35 injury determ...   
8   ab finding apply claim ada art 93 gatt art vi2...   
9   finding 2 ada art 651ascm art 1241 evidence co...   
10  findings gpa art scope koreas gpa appendix com...   
11  ab finding ascm art 11a1 definition public bod...   
12  ab findings2 ascm art 11a1 definition public b...   
13  finding specificity panel uphold several claim...   
14  ab findings sunset review ada art 113 continua...   

                                    Summary_segmented  
0   [findings2 ascm art 14d rej

In [7]:
# Save the dataset to a new CSV file
df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"\n✅ Processed dataset saved successfully to: {PROCESSED_DATA_PATH}")



✅ Processed dataset saved successfully as 'Pre-processed_database.csv'!


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
df["Title_embedding"] = df["Title_tokenized"].apply(lambda x: model.encode(x) if pd.notnull(x) else None)
df["Summary_embedding"] = df["Summary_lemmatized"].apply(lambda x: model.encode(x) if pd.notnull(x) else None)
df["Articles_embedding"] = df["Articles_lemmatized"].apply(lambda x: model.encode(x) if pd.notnull(x) else None)

# Print results
print(df[["Title_embedding", "Articles_embedding", "Summary_embedding"]].head())


                                     Title_embedding  \
0  [-0.11275301, 0.06361867, -0.0030210014, 0.027...   
1  [-0.05017399, 0.031692185, 0.016254203, 0.0076...   
2  [0.02813397, 0.008265634, 0.0011514709, 0.0319...   
3  [-0.043207664, 0.020380212, -0.07205802, 0.006...   
4  [-0.11614925, 0.002922292, -0.030505868, 0.023...   

                                  Articles_embedding  \
0  [-0.04005064, 0.010635478, -0.07270459, 0.0211...   
1  [-0.08458803, -0.07789096, -0.063465476, 0.020...   
2  [-0.10868006, 0.095001034, 0.02837096, 0.02743...   
3  [-0.08587139, -0.0065624523, -0.038671054, 0.0...   
4  [-0.12557751, -0.023380494, -0.016130986, -0.0...   

                                   Summary_embedding  
0  [-0.017498909, -0.029531453, -0.02912183, 0.00...  
1  [-0.070989706, -0.04217409, -0.02386628, 0.025...  
2  [-0.005143401, -0.03873782, 0.0045264647, -0.0...  
3  [-0.026871117, -0.08064937, -0.024373632, 0.01...  
4  [-0.023795277, -0.04733631, -0.013429214, -0.0..

In [9]:
# Function to process user queries (same as dataset cleaning)
def preprocess_query(query):
    query = normalize_text(query)
    query = tokenize_and_remove_stopwords(query)
    query = lemmatize_text(query)
    return query

# Example query processing
user_query = "What are the legal precedents for trade disputes?"
processed_query = preprocess_query(user_query)
query_embedding = model.encode(processed_query)

# Print query results
print(f"Original Query: {user_query}")
print(f"Processed Query: {processed_query}")
print(f"Query Embedding (first 5 values): {query_embedding[:5]}")


Original Query: What are the legal precedents for trade disputes?
Processed Query: legal precedent trade dispute
Query Embedding (first 5 values): [-0.09569442  0.02848781  0.02377198 -0.1077167  -0.02603052]


In [10]:
import numpy as np

# Save embeddings as .npy (NumPy format, efficient for loading later)
np.save(PROCESSED_DIR / "title_embeddings.npy", np.array(df["Title_embedding"].tolist()))
np.save(PROCESSED_DIR / "summary_embeddings.npy", np.array(df["Summary_embedding"].tolist()))
np.save(PROCESSED_DIR / "articles_embeddings.npy", np.array(df["Articles_embedding"].tolist()))

print("\n✅ Embeddings saved successfully as NumPy files!")



✅ Embeddings saved successfully as NumPy files!
